In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
import healpy as hp

In [ ]:
import rubin_sim.maf as maf
import rubin_sim.utils as rsUtils
from rubin_sim.data import get_baseline
import rubin_sim.maf.db as db

## Set up and run the microlensing metric. ## 

The microlensing metric is similar to the KNe metric, in that it requires a microlensing-specific slicer (a UserPoints slicer combined with information about the microlensing events and lightcurves) together with the microlensing detection metric. 

The population distribution for the microlensing metric is currently a simple N^2 distribution of events (N being the stellar density at that point in the sky). Events occur with a variety of crossing times, and evaluation of the detection likelilhood in the metric is split up between different intervals; in general longer crossing times are easier to detect and can overwhelm the detection sensitivity to shorter timescale events.

In [ ]:
baseline_file = get_baseline()
opsim = os.path.basename(baseline_file).replace(".db", "")
print(f"running on {opsim}")

In [ ]:
metric = maf.MicrolensingMetric()
summaryMetrics = maf.batches.lightcurve_summary()

In [ ]:
n_events = 10000
bundles = {}
resultDbs = {}
# Let's evaluate a variety of crossing times
crossing_times = [
    [
        1,
        5,
    ],
    [5, 10],
    [10, 20],
    [20, 30],
    [30, 60],
    [60, 90],
    [100, 200],
    [200, 500],
    [500, 1000],
]
for crossing in crossing_times:
    key = f"{crossing[0]} to {crossing[1]}"
    slicer = maf.generate_microlensing_slicer(
        min_crossing_time=crossing[0], max_crossing_time=crossing[1], n_events=n_events
    )
    bundles[key] = maf.MetricBundle(
        metric,
        slicer,
        None,
        run_name=opsim,
        summary_metrics=summaryMetrics,
        info_label=f"tE {crossing[0]}_{crossing[1]} days",
    )

In [ ]:
outDir = "test_microlensing"
g = maf.MetricBundleGroup(bundles, baseline_file, outDir)

In [ ]:
g.run_all()

In [ ]:
bundles.keys()

## Running Number of Points microlensing metric

It may also be useful as an estimate of characterization efficiency to calculate the number of points observed close to the peak of the event. When you specify metricCalc = 'Npts', the metric calculates the total number of points within two Einstein crossing times of the peak of the event

In [ ]:
metric_Npts = maf.MicrolensingMetric(metric_calc="Npts")
summaryMetrics = maf.batches.microlensing_summary(metric_type="Npts")

In [ ]:
n_events = 10000
bundles_Npts = {}
# Let's evaluate a variety of crossing times
crossing_times = [
    [
        1,
        5,
    ],
    [5, 10],
    [10, 20],
    [20, 30],
    [30, 60],
    [60, 90],
    [100, 200],
    [200, 500],
    [500, 1000],
]
for crossing in crossing_times:
    key = f"{crossing[0]} to {crossing[1]}"
    slicer = maf.generate_microlensing_slicer(
        min_crossing_time=crossing[0], max_crossing_time=crossing[1], n_events=n_events
    )
    bundles_Npts[key] = maf.MetricBundle(
        metric_Npts,
        slicer,
        None,
        run_name=opsim,
        summary_metrics=summaryMetrics,
        info_label=f"tE {crossing[0]}_{crossing[1]} days",
    )

In [ ]:
outDir = "test_microlensing"
g = maf.MetricBundleGroup(bundles_Npts, baseline_file, outDir)

In [ ]:
g.run_all()

## Running Fisher Information Matrix microlensing metric

In order to find the characterization efficiency more exactly, we can calculate the Fisher information matrix and find the $\sigma_{t_E}/{t_E}$ (where $t_E$ is the Einstein crossing time) for each event. Note that this is more computationally expensive

In [ ]:
import imp

imp.reload(maf.batches.common)

In [ ]:
metric_Fisher = maf.MicrolensingMetric(metric_calc="Fisher")
summaryMetrics = maf.batches.microlensing_summary(metric_type="Fisher")

In [ ]:
n_events = 10000
bundles_Fisher = {}
# Let's evaluate a variety of crossing times
crossing_times = [
    [1, 5],
    [5, 10],
    [10, 20],
    [20, 30],
    [30, 60],
    [60, 90],
    [100, 200],
    [200, 500],
    [500, 1000],
]
for crossing in crossing_times:
    key = f"{crossing[0]} to {crossing[1]}"
    slicer = maf.generate_microlensing_slicer(
        min_crossing_time=crossing[0], max_crossing_time=crossing[1], n_events=n_events
    )
    bundles_Fisher[key] = maf.MetricBundle(
        metric_Fisher,
        slicer,
        None,
        run_name=opsim,
        summary_metrics=summaryMetrics,
        info_label=f"tE {crossing[0]}_{crossing[1]} days",
    )

In [ ]:
outDir = "test_microlensing"
g = maf.MetricBundleGroup(bundles_Fisher, baseline_file, outDir)

In [ ]:
g.run_all()

## Look at the metric outputs. ##

The default microlensing metric can output two different kinds of results; if 'detect' is True (the default), then it outputs 0-1 depending on whether an event was detected or not.

In [ ]:
# If you don't want to try and plot N individual points,
plotDict = {"reduce_func": np.sum, "nside": 64, "color_min": 0, "color_max": 20}
plotFunc = maf.plots.HealpixSkyMap()
ph = maf.plots.PlotHandler(out_dir=outDir, figformat="png", thumbnail=False)
for k in bundles:
    ph.set_metric_bundles([bundles[k]])
    ph.plot(plot_func=plotFunc, plot_dicts=plotDict)

In [ ]:
# If you do want to show each individual point - this is slower but shows each individual event
plotFunc = maf.plots.BaseSkyMap()

ph = maf.plots.PlotHandler(out_dir=outDir, figformat="png", thumbnail=False)
for k in bundles:
    ph.set_metric_bundles([bundles[k]])
    ph.plot(plot_func=plotFunc)

In [ ]:
d = pd.DataFrame([bundles[k].summary_values for k in bundles.keys()], index=list(bundles.keys()))
d

In [ ]:
plt.plot(d["Total detected"])
plt.xticks(rotation=90)
plt.title(f"Total detected / {n_events} as function of crossing time")

In [ ]:
# Illustrate some things about the metric values: (caught in summary metrics)
m = list(bundles.keys())[-1]
print(f"How many lightcurves were added? (over the entire sky) {len(bundles[m].metric_values)}")
print(
    f"How many lightcurves were added in areas that were part of the survey footprint?",
    f"{len(bundles[m].metric_values.compressed())}",
)
print(
    f"What are the metric values for each of these light curves? "
    f"{np.unique(bundles[m].metric_values.compressed())}"
)
print(f"How many lightcurves were *successfully* detected? {bundles[m].metric_values.sum()}")
print(len(np.where(bundles[m].metric_values == 1)[0]))
frac_total = bundles[m].metric_values.sum() / len(bundles[m].metric_values)
print(f"Fraction of total lightcurves detected {frac_total}")
frac_footprint = bundles[m].metric_values.sum() / len(bundles[m].metric_values.compressed())
print(f"Fraction of lightcurves within footprint detected {frac_footprint}")

In [ ]:
d_Npts = pd.DataFrame(
    [bundles_Npts[k].summary_values for k in bundles_Npts.keys()], index=list(bundles_Npts.keys())
)
d_Npts

In [ ]:
d_Fisher = pd.DataFrame(
    [bundles_Fisher[k].summary_values for k in bundles_Fisher.keys()], index=list(bundles_Fisher.keys())
)
d_Fisher

## Preparing metric results and creating figure of merit and comparison plots

In [ ]:
results = []
results_compare = []
run_names = []
metric_types = []
min_tEs = []
max_tEs = []

bundle_dicts = [bundles, bundles_Npts, bundles_Fisher]

for bundle_dict in range(len(bundle_dicts)):
    if bundle_dict == 0:  # detect metric
        results.extend(d["Fraction detected of total (mean)"])
        # results_compare.extend(d['Fraction detected of total (mean)'])
        metric_types.extend(["detect"] * len(bundles.keys()))
    elif bundle_dict == 1:  # Npts metric
        results.extend(d_Npts["Mean number of points per lightcurves in total"])
        # results_compare.extend(d_Npts['Fraction w/ at least 10 points'])
        metric_types.extend(["Npts"] * len(bundles_Npts.keys()))
    elif bundle_dict == 2:  # Fisher metric
        results.extend(d_Fisher["Fraction w/ sigma_tE/tE < 0.1"])
        # results_compare.extend(d_Fisher['Fraction w/ sigma_tE/tE < 0.1'])
        metric_types.extend(["Fisher"] * len(bundles_Fisher.keys()))
    for i in bundle_dicts[bundle_dict]:
        results_compare.append(bundle_dicts[bundle_dict][i].metric_values)
        run_names.append(bundle_dicts[bundle_dict][i].run_name)
        min_tEs.append(np.int(i.split(" ")[0]))
        max_tEs.append(np.int(i.split(" ")[2]))

results = np.array(results)
results_compare = np.array(results_compare)
run_names = np.array(run_names)
metric_types = np.array(metric_types)
min_tEs = np.array(min_tEs)
max_tEs = np.array(max_tEs)

The following figure is intended to be used to compare between different OpSims

In [ ]:
save_folder = "test_microlensing"
figure_name = "microlensingFOM"
figsize = (30, 5)  # Increase 5 for more metrics
maf.plot_fom(results, run_names, metric_types, min_tEs, max_tEs, save_folder, figure_name, figsize=figsize)

The following set of figures compares between the discovery/detect metric, number of points metric, and characterization/fisher metric. By default the fraction discovered, the fraction of objects with at least 10 points within 2tE of the peak, and the fraction characterized are compared. The three should all be correlated, but these plots are helpful for knowing how much followup must be done outside of Rubin to characterize events.

In [ ]:
maf.plot_compare(results_compare, run_names, metric_types, min_tEs, max_tEs, save_folder)

## Look at the slicer information (how the lightcurves were added). ##

In [ ]:
# The *slicer* keeps the information about the injected lightcurves too
slicer = bundles[m].slicer
print(f"How many lightcurves added over the sky? {len(slicer)}")

In [ ]:
# What information was recorded for each event
slicer.slice_points.keys()

In [ ]:
# Including their spatial, time and distance distribution
hp.mollview(
    rsUtils._healbin(
        slicer.slice_points["ra"],
        slicer.slice_points["dec"],
        slicer.slice_points["peak_time"],
        64,
        reduce_func=np.mean,
    ),
    unit="peak time (days)",
    title="Lightcurve Peak Times",
    min=0,
    max=3650,
)